## В этом домашнем задании вы сделаете первые шаги в мире линейной бинарной классификации!

В этом задании мы будем работать с данными Microsoft Malware Detection.

Таргетом будет последний столбец HasDetections, который принимает значения {0, 1} в случае отсутствия или наличия вируса на компьютере соответственно. Признаками будут выступать всевозможные характеристики компьютера.

Для начала давайте посмотрим на соотношение классов в таргете. Удалите константные признаки, признаки ProductName, MachineIdentifier и выведите соотношение классов в таргете через запятую и пробел. Сначала positive class, затем negative class.

Пример ответа: 10006, 999

In [ ]:
import pandas as pd
import numpy as np
import warnings

pd.options.display.max_columns = 500



In [ ]:
data = pd.read_csv('train.csv')


In [ ]:
data.head()

,MachineIdentifier,ProductName,EngineVersion,AppVersion,AvSigVersion,IsBeta,RtpStateBitfield,IsSxsPassiveMode,DefaultBrowsersIdentifier,AVProductStatesIdentifier,...,Census_FirmwareVersionIdentifier,Census_IsSecureBootEnabled,Census_IsWIMBootEnabled,Census_IsVirtualDevice,Census_IsTouchEnabled,Census_IsPenCapable,Census_IsAlwaysOnAlwaysConnectedCapable,Wdft_IsGamer,Wdft_RegionIdentifier,HasDetections
0,0000028988387b115f69f31a3bf04f09,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1735.0,0,7.0,0,NaN,53447.0,...,36144.0,0,NaN,0.0,0,0,0.0,0.0,10.0,0
1,000007535c3f730efa9ea0b7ef1bd645,win8defender,1.1.14600.4,4.13.17134.1,1.263.48.0,0,7.0,0,NaN,53447.0,...,57858.0,0,NaN,0.0,0,0,0.0,0.0,8.0,0
2,000007905a28d863f6d0d597892cd692,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1341.0,0,7.0,0,NaN,53447.0,...,52682.0,0,NaN,0.0,0,0,0.0,0.0,3.0,0
3,00000b11598a75ea8ba1beea8459149f,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1527.0,0,7.0,0,NaN,53447.0,...,20050.0,0,NaN,0.0,0,0,0.0,0.0,3.0,1
4,000014a5f00daa18e76b81417eeb99fc,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1379.0,0,7.0,0,NaN,53447.0,...,19844.0,0,0.0,0.0,0,0,0.0,0.0,1.0,1


Удалите константные признаки и признаки `ProductName` `MachineIdentifier`

In [ ]:
# Можно воспользоваться VarianceThreshold или можно еще проще с помощью скрина метода .all()

for i in data.columns:
    
    if (data[i] == data[i][0]).all():
        data.drop(columns=[i], inplace=True)

In [ ]:
# Дополнительно удалить еще 2 колонки

data.drop(columns=['ProductName', 'MachineIdentifier'], inplace=True)

In [ ]:
# Сколько наблюдений в датасете имеют 0 и 1 классы

print(sum(data['HasDetections'] == 1), '- positive class,')
print(sum(data['HasDetections'] == 0), '- negative class')



100060 - positive class,
99940 - negative class


Избавьтесь от пропусков в данных! Новый для нас прием: признаки с более чем половиной пропусков следует удалить.

Согласитесь, если в вашей колоночке среди 100 объектов всего лишь у 2 есть какое-то не пропущенное значение, странно все остальные заполнять средним от этих двух чисел. Такие "редкие" признаки лучше вообще опустить! В категориальных колонках заменим отсутствующую категорию просто некоторой новой и назовем ее NaN. А в числовых, ради разнообразия, заполним пропуски медианным значением. Создайте копию полученного датафрейма и положите ее в переменную data_2.

Введите количество колонок в полученном датафрейме data_2.

In [ ]:
# Посчитать количество отсутствующих значений в каждой колонке и удалить те колонки, в которых пропусков более половины от размера всей выборки

nans = np.sum(data.isna(), axis=0)


data.drop(columns=list(nans[nans > data.shape[0] / 2].index), inplace=True)

In [ ]:
# В категориальных колонках с отсутствующими значениями произведем fillna, создав по новой категории для каждой такой фичи

cat_cols = data.select_dtypes(exclude=['float64', 'int64']).isnull().any()

for null_col in cat_cols[cat_cols].index:
    data[null_col] = data[null_col].fillna('NaN')


In [ ]:
# Оставшиеся (вещественные) колонки заполним медианой по столбцу

nans = np.sum(data.isna(), axis=0)


for col in list(nans[nans > 0].index):
    data[col].fillna(data[col].median(), inplace=True)

In [10]:
data.head()

,EngineVersion,AppVersion,AvSigVersion,RtpStateBitfield,IsSxsPassiveMode,AVProductStatesIdentifier,AVProductsInstalled,AVProductsEnabled,HasTpm,CountryIdentifier,...,Census_FirmwareManufacturerIdentifier,Census_FirmwareVersionIdentifier,Census_IsSecureBootEnabled,Census_IsVirtualDevice,Census_IsTouchEnabled,Census_IsPenCapable,Census_IsAlwaysOnAlwaysConnectedCapable,Wdft_IsGamer,Wdft_RegionIdentifier,HasDetections
0,1.1.15100.1,4.18.1807.18075,1.273.1735.0,7.0,0,53447.0,1.0,1.0,1,29,...,628.0,36144.0,0,0.0,0,0,0.0,0.0,10.0,0
1,1.1.14600.4,4.13.17134.1,1.263.48.0,7.0,0,53447.0,1.0,1.0,1,93,...,628.0,57858.0,0,0.0,0,0,0.0,0.0,8.0,0
2,1.1.15100.1,4.18.1807.18075,1.273.1341.0,7.0,0,53447.0,1.0,1.0,1,86,...,142.0,52682.0,0,0.0,0,0,0.0,0.0,3.0,0
3,1.1.15100.1,4.18.1807.18075,1.273.1527.0,7.0,0,53447.0,1.0,1.0,1,88,...,355.0,20050.0,0,0.0,0,0,0.0,0.0,3.0,1
4,1.1.15100.1,4.18.1807.18075,1.273.1379.0,7.0,0,53447.0,1.0,1.0,1,18,...,355.0,19844.0,0,0.0,0,0,0.0,0.0,1.0,1


Создайте копию полученного датафрейма и положите ее в переменную data_2. Понадобится в следующих заданиях.

In [11]:
data_2 = data.copy()

In [ ]:
# Для OneHotEncoding'а воспользуемся пандасовским get_dummies

cat_cols = data.select_dtypes(exclude=['float64', 'int64']).columns

categorical_data_part = pd.get_dummies(data[cat_cols],
                                       prefix=cat_cols,
                                       drop_first=True,
                                       )

data = data.drop(cat_cols, axis=1)

data = pd.concat((data, categorical_data_part), axis=1)

Разделим выборку на тренировочную и тестовую

В задачах классификации, как и в задаче регрессии, можно использовать технологию Кросс-Валидации. В этом задании попросим Вас реализовать технологию отложенной выборки в пропорции 3:1.

Соберите Pipeline, реализовав в нем 2 шага: стандартизация данных через StandardScaler и обучение логистической регрессии с помощью LogisticRegression, положите результаты в переменную pipe, а в классе модели LogisticRegression укажите параметр penalty='none'.

Далее, чтобы замерить качество работы такой модели на трейне и на тесте воспользуемся функцией cross_validate. Вопрос: что передавать ей в параметр cv? Ведь мы уже разделили нашу выборку на трейн и тест (имеем всего 1 fold). Для этого можно просто передать список от кортежа, содержащего индексы тренировочных и тестовых объектов.

Введите точность модели на тестовых данных, округлите до трех знаков после точки. В качестве десятичного разделителя используйте точку.

In [ ]:
# Разобить выборку на трейн и тест

from sklearn.model_selection import train_test_split 

X = data.drop(columns=['HasDetections'])
y = data['HasDetections']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    shuffle=True,
                                                    random_state=1)

Соберите `Pipeline`, реализовав в нем 2 шага: стандартизация данных через `StandardScaler` и обучение логистической регрессии с помощью `LogisticRegression`, положите результаты в переменную `pipe`, а в классе модели `LogisticRegression` укажите параметр `penalty='none'`

In [14]:
### Здесь все просто и знакомо

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline([('scaler', StandardScaler()), 
                 ('LR', LogisticRegression(penalty='none'))])

Чтобы замерить качество работы такой модели на трейне и на тесте воспользуемся функцией `cross_validate`

Вопрос: что передавать ей в параметр cv? Ведь мы уже разделили нашу выборку на трейн и тест (имеем всего 1 fold). Для этого можно просто передать список от кортежа, содержащего индексы тренировочных и тестовых объектов.

In [ ]:
from sklearn.model_selection import cross_validate
import datetime

# Создадать такой список из одного единственного кортежа, первый элемент - индексы обучающей выборки. Второй элемент - индексы тестовой выборки

custom_cv = [(X_train.index.to_list(), X_test.index.to_list())]

# Замерить начало работы cross_validate

begin_time = datetime.datetime.now()

# В cv передадим созданный custom_cv. В качестве выборки передадим все данные. Метод сам разобьет выборку на трейн и тест по нашей стратегии

cv_result_pipe = cross_validate(pipe, X, y, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)


print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

Теперь попробуем другой метод кодирования категориальных колонок, а именно счётчики. Построем ту же модель и на том же разделении, просто заново иначе переобработаем датасет.

Для тех категориальных признаков, у которых количество уникальных значений в колонках больше 5, применим MeanTargetEncoding. Для остальных оставим любимый OneHotEncoding.

В качестве ответа напишите точность модели на тестовых данных, округлите до трех знаков после точки. В качестве десятичного разделителя используйте точку.

In [ ]:
# Закодировать категории с большим кол-ом уникальных значений с помощью счетчиков закрывая глаза на возможную проблему переобучения
### То есть будем считать средние по всему датасету

for col in cat_cols:
    if data_2[col].nunique() <= 5:
        one_hot = pd.get_dummies(data_2[col], prefix=col, drop_first=True)
        data_2 = pd.concat((data_2.drop(col, axis=1), one_hot), axis=1)
        
    else:
        mean_target = data_2.groupby(col)['HasDetections'].mean()
        data_2[col] = data_2[col].map(mean_target)

In [17]:
X_2 = data_2.drop(columns=['HasDetections'])
y_2 = data_2['HasDetections']

In [ ]:
### Заново обучим модель и провалидируем


begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(pipe, X_2, y_2, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)

print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

Accuracy на трейне: 0.638
Accuracy на тесте: 0.638
Время работы алгоритма: 0:00:02.645526


Как и в моделях регрессии, решая задачу классификации, можем штрафовать минимизируемый функционал за большие веса, добавив к нему L1 или L2 норму весов.

Для этого в изначальном классе LogisticRegression изменить параметр penalty на l1 или l2 соответственно. Выберите второй вариант! Можно воспользоваться методом set_params и применить его к pipe.

Теперь наша модель будет строить логистическую регрессию с L2 регуляризатором! Помним, что у регуляризируемых моделей есть гиперпараметр, контролирующий силу регуляризации, который выбирается ДО запуска метода fit, то есть заранее, когда модель еще не обучена. Конечно же, выбор этого параметра влияет на итоговые результаты. Хочется поперебирать различные параметры регуляризации и найти такой, при котором качество на тесте окажется лучшим! Сгенерируйте массив гиперпараметров и отберите среди массива лучший, используя конструкцию GridSearchCV из sklearn.

Выведите точность лучшего параметра, округлив значение до трех знаков после точки. В качестве десятичного разделителя используйте точку.

In [19]:
### Установим в качестве параметра penalty
### L2 регуляризатор!

pipe.set_params(LR__penalty='l2')

Pipeline(steps=[('scaler', StandardScaler()), ('LR', LogisticRegression())])

In [20]:
### Сгенерим кучу параметров для регуляризации

alphas = np.linspace(0.01, 100, 100)
alphas

array([1.000e-02, 1.020e+00, 2.030e+00, 3.040e+00, 4.050e+00, 5.060e+00,
       6.070e+00, 7.080e+00, 8.090e+00, 9.100e+00, 1.011e+01, 1.112e+01,
       1.213e+01, 1.314e+01, 1.415e+01, 1.516e+01, 1.617e+01, 1.718e+01,
       1.819e+01, 1.920e+01, 2.021e+01, 2.122e+01, 2.223e+01, 2.324e+01,
       2.425e+01, 2.526e+01, 2.627e+01, 2.728e+01, 2.829e+01, 2.930e+01,
       3.031e+01, 3.132e+01, 3.233e+01, 3.334e+01, 3.435e+01, 3.536e+01,
       3.637e+01, 3.738e+01, 3.839e+01, 3.940e+01, 4.041e+01, 4.142e+01,
       4.243e+01, 4.344e+01, 4.445e+01, 4.546e+01, 4.647e+01, 4.748e+01,
       4.849e+01, 4.950e+01, 5.051e+01, 5.152e+01, 5.253e+01, 5.354e+01,
       5.455e+01, 5.556e+01, 5.657e+01, 5.758e+01, 5.859e+01, 5.960e+01,
       6.061e+01, 6.162e+01, 6.263e+01, 6.364e+01, 6.465e+01, 6.566e+01,
       6.667e+01, 6.768e+01, 6.869e+01, 6.970e+01, 7.071e+01, 7.172e+01,
       7.273e+01, 7.374e+01, 7.475e+01, 7.576e+01, 7.677e+01, 7.778e+01,
       7.879e+01, 7.980e+01, 8.081e+01, 8.182e+01, 

Чтобы отобрать среди данного массива гиперпараметров лучший, воспользуйтесь конструкцией `GridSearchCV` из `sklearn`

In [21]:
from sklearn.model_selection import GridSearchCV

param_grid ={
    'LR__C': alphas
}


search_alpha = GridSearchCV(pipe, param_grid, 
                            cv=custom_cv, scoring='accuracy')

search_alpha.fit(X_2, y_2)

print(f"Best parameter (CV score={search_alpha.best_score_:.5f}):")
print(search_alpha.best_params_)

Best parameter (CV score=0.63822):
{'LR__C': 0.01}


In [152]:
from sklearn.model_selection import GridSearchCV

X_new = data_new.drop(columns=['HasDetections'])
y_new = data_new['HasDetections']

param_grid ={
    'LR__C': alphas
}

search_alpha = GridSearchCV(pipe, param_grid, 
                            cv=custom_cv, scoring='accuracy')

search_alpha.fit(X_new, y_new)

print(f"Best parameter (CV score={search_alpha.best_score_:.5f}):")
print(search_alpha.best_params_)

Best parameter (CV score=0.79430):
{'LR__C': 0.01}
